In [3]:
import pandas as pd
import numpy as np
import folium

In [4]:
from sqlalchemy import create_engine
from urllib.parse import quote_plus

In [5]:
MYSQL_HOST = "localhost"
MYSQL_PORT = 3306
MYSQL_USER = "root"
MYSQL_PASSWORD = "Samira@8144"
MYSQL_DATABASE = "olist_ecommerce"

engine = create_engine(
    f"mysql+mysqlconnector://{MYSQL_USER}:{quote_plus(MYSQL_PASSWORD)}@{MYSQL_HOST}:{MYSQL_PORT}/{MYSQL_DATABASE}"
)

print("Database connected successfully")

Database connected successfully


In [6]:
monthly_revenue = pd.read_sql("""
SELECT
    DATE_FORMAT(o.order_purchase_timestamp,'%Y-%m') AS month,
    ROUND(SUM(p.payment_value),2) AS revenue
FROM orders o
JOIN order_payments p
    ON o.order_id = p.order_id
WHERE o.order_status='delivered'
GROUP BY month
ORDER BY month
""", engine)

monthly_revenue = monthly_revenue[
    monthly_revenue["month"] < "2018-09"
]

monthly_revenue.to_csv(
    "../data/processed/monthly_revenue.csv",
    index=False
)

In [7]:
payment_methods = pd.read_sql("""
SELECT
    payment_type,
    COUNT(*) AS transactions,
    ROUND(SUM(payment_value),2) AS total_value
FROM order_payments
GROUP BY payment_type
""", engine)

payment_methods.to_csv(
    "../data/processed/payment_methods.csv",
    index=False
)

In [8]:
top_categories = pd.read_sql("""
SELECT
    ct.product_category_name_english AS category,
    ROUND(SUM(oi.price),2) AS revenue
FROM order_items oi
JOIN products p
    ON oi.product_id = p.product_id
JOIN category_translation ct
    ON p.product_category_name = ct.product_category_name
GROUP BY category
ORDER BY revenue DESC
""", engine)

top_categories.to_csv(
    "../data/processed/top_categories.csv",
    index=False
)

In [9]:
revenue_by_state = pd.read_sql("""
SELECT
    c.customer_state,
    ROUND(SUM(pay.payment_value),2) AS revenue
FROM customers c
JOIN orders o
    ON c.customer_id = o.customer_id
JOIN order_payments pay
    ON o.order_id = pay.order_id
GROUP BY c.customer_state
ORDER BY revenue DESC
""", engine)

revenue_by_state.to_csv(
    "../data/processed/revenue_by_state.csv",
    index=False
)

In [10]:
kpi_summary = pd.DataFrame({
    "metric": [
        "Total Revenue",
        "Total Orders",
        "Champion Customers"
    ],
    "value": [
        16008872,
        99441,
        1861
    ]
})

kpi_summary.to_csv(
    "../data/processed/kpi_summary.csv",
    index=False
)